In [ ]:
import h5py
import numpy as np
import scipy.io as sio

# 1. Carga del archivo

def load_ninapro_file(file_path):
    data_dict = {}
    try:
        mat_data = sio.loadmat(file_path)
        for key in ["emg", "stimulus", "restimulus", "repetition"]:
            if key in mat_data:
                data_dict[key] = np.squeeze(mat_data[key])
    except (NotImplementedError, OSError):
        with h5py.File(file_path, "r") as f:
            for key in ["emg", "stimulus", "restimulus", "repetition"]:
                if key in f:
                    arr = np.array(f[key])
                    if arr.ndim > 1 and arr.shape[0] < arr.shape[1]:
                        arr = arr.T
                    data_dict[key] = np.squeeze(arr)

    # Priorizar 'restimulus'
    labels = data_dict.get("restimulus", data_dict.get("stimulus"))
    repetition = data_dict.get("repetition")
    return data_dict["emg"], labels, repetition


# 2. Funciones de características en el dominio del tiempo
def mean_absolute_value(x):
    """MAV: Magnitud media absoluta."""
    return np.mean(np.abs(x), axis=0)


def root_mean_square(x):
    """RMS: Valor cuadrático medio."""
    return np.sqrt(np.mean(x**2, axis=0))


def waveform_length(x):
    """WL: Longitud acumulada de la onda."""
    return np.sum(np.abs(np.diff(x, axis=0)), axis=0)


def variance(x):
    """VAR: Varianza de la señal."""
    return np.var(x, axis=0)


def zero_crossings(x, threshold=1e-4):
    """ZC: Cruces por cero que superen un umbral de ruido."""
    diff = x[:-1, :] * x[1:, :]
    mag_diff = np.abs(x[:-1, :] - x[1:, :])
    return np.sum((diff < 0) & (mag_diff >= threshold), axis=0)


def slope_sign_changes(x, threshold=1e-4):
    """SSC: Cambios de signo en la pendiente."""
    diff1 = x[1:-1, :] - x[:-2, :]
    diff2 = x[1:-1, :] - x[2:, :]
    return np.sum(
        (diff1 * diff2 > 0)
        & ((np.abs(diff1) >= threshold) | (np.abs(diff2) >= threshold)),
        axis=0,
    )


# 3. Segmentación en ventanas y extracción de matriz de características

def extract_time_features(emg, labels, repetition, win_len=200, win_step=50, threshold=1e-4):
    """Segmenta la señal en ventanas deslizantes y extrae el vector de características.

    Parámetros:
      - win_len: tamaño de ventana en muestras (ej. 200 ms a 1 kHz = 200 muestras).
      - win_step: desplazamiento de la ventana (ej. 50 ms de solapamiento = 50 muestras).
      - threshold: umbral de ruido para ZC y SSC.

    Retorna:
      - X: matriz (n_ventanas, n_canales * n_features).
      - y: vector (n_ventanas,) con la etiqueta predominante (moda) de cada ventana.
    """
    n_samples, n_channels = emg.shape
    num_windows = (n_samples - win_len) // win_step + 1

    features_list = []
    window_labels = []
    window_reps = [] # repeticiones de la ventana

    for i in range(num_windows):
        start = i * win_step
        end = start + win_len

        window_emg = emg[start:end, :]
        window_lbl = labels[start:end]
        window_rep = repetition[start:end]

        # Extraer características para todos los canales en esta ventana
        mav = mean_absolute_value(window_emg)
        rms = root_mean_square(window_emg)
        wl = waveform_length(window_emg)
        var = variance(window_emg)
        zc = zero_crossings(window_emg, threshold=threshold)
        ssc = slope_sign_changes(window_emg, threshold=threshold)

        # Concatenar características: shape = (n_canales * 6,)
        feat_vector = np.concatenate([mav, rms, wl, var, zc, ssc])
        features_list.append(feat_vector)

        # Asignar a la ventana la clase mayoritaria (moda)
        vals, counts = np.unique(window_lbl, return_counts=True)
        window_labels.append(vals[np.argmax(counts)])

        vals_rep, counts_rep = np.unique(window_rep, return_counts=True)
        window_reps.append(vals_rep[np.argmax(counts_rep)])

    X = np.array(features_list)
    y = np.array(window_labels)
    groups = np.array(window_reps)  # Repeticiones array
    return X, y, groups


# -------------------------------------------------------------------------
# 4. Ejecución del pipeline
# -------------------------------------------------------------------------
if __name__ == "__main__":
    
    from lazypredict.Supervised import LazyClassifier
    from sklearn.model_selection import train_test_split

    file_path = "S1_A1_E1.mat"

    # Cargar datos
    emg, labels, repetition = load_ninapro_file(file_path)

    WIN_LEN = 200  # Ajustar según frecuencia de muestreo
    WIN_STEP = 50  # Solapamiento del 75%

    X, y, groups = extract_time_features(
        emg, labels, repetition, win_len=WIN_LEN, win_step=WIN_STEP, threshold=1e-4
    )

    print("\n--- Extracción completada ---")

    # ---------------------------------------------------------
    # MÉTODO 1: train_test_split tradicional (Aleatorio)
    # ---------------------------------------------------------

    X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
        X, y, test_size=0.3, random_state=123, stratify=y
    )

    print("\nEntrenando LazyPredict con Split Aleatorio...")
    clf_rand = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
    models_rand, _ = clf_rand.fit(X_train_rand, X_test_rand, y_train_rand, y_test_rand)

    # ---------------------------------------------------------
    # MÉTODO 2: Leave-Repetitions-Out (Entrenar con [1,3,4,6,7,8,9,10], Testear con [2,5])
    # ---------------------------------------------------------

    
    reps_test = [2, 5] 
    test_mask = np.isin(groups, reps_test)
    train_mask = ~test_mask 

    X_train_rep = X[train_mask]
    y_train_rep = y[train_mask]
    
    X_test_rep = X[test_mask]
    y_test_rep = y[test_mask]

    print("\nEntrenando LazyPredict con Leave-Repetitions-Out (Test = reps 2 y 5)...")
    clf_rep = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
    models_rep, _ = clf_rep.fit(X_train_rep, X_test_rep, y_train_rep, y_test_rep)

    # ---------------------------------------------------------
    # resultados
    # ---------------------------------------------------------
    print("\n" + "="*50)
    print("RESULTADOS 1: SPLIT ALEATORIO (30% Test)")
    print("="*50)
    print(models_rand.head(10)) # Mostramos solo el top 10 para no saturar

    print("\n" + "="*50)
    print("RESULTADOS 2: SPLIT POR REPETICIONES (Test = Reps 2 y 5)")
    print("="*50)
    print(models_rep.head(10))


--- Extracción completada ---

Entrenando LazyPredict con Split Aleatorio...

Entrenando LazyPredict con Leave-Repetitions-Out (Test = reps 2 y 5)...

RESULTADOS 1: SPLIT ALEATORIO (30% Test)
                            Accuracy  Balanced Accuracy   ROC AUC  F1 Score  \
Model                                                                         
ExtraTreesClassifier        0.914191           0.846332  0.984889  0.912682   
LinearSVC                   0.909241           0.830863       NaN  0.905927   
LabelPropagation            0.877888           0.827722       NaN  0.878329   
LabelSpreading              0.877888           0.827722       NaN  0.878329   
LogisticRegression          0.897690           0.815983  0.973310  0.893693   
RandomForestClassifier      0.904290           0.812682  0.981250  0.899474   
LinearDiscriminantAnalysis  0.871287           0.805968  0.955071  0.869593   
BaggingClassifier           0.909241           0.805604  0.968013  0.906161   
KNeighborsClassif